# Préparation des données — Modèle CLV Nouvelles Acquisitions

Ce notebook lit les 3 sources de données disponibles, les fusionne et produit le DataFrame `df` utilisé par toutes les analyses d'hypothèses (HA1 à HA10).

## Sources de données attendues dans `data/`
| Fichier | Contenu |
|---------|--------|
| `STOC_23.csv`, `STOC_24.csv`, `STOC_25.csv` | Démographiques clients : date_creation, segment, profession |
| `PNB_23.csv`, `PNB_24.csv`, `PNB_25.csv` | NBI mensuel par client — colonnes = indices mois bancaires (ex: 349 = jan 2023) |
| `clients_equipement_full.csv` | Équipements produits trimestriels (compte_q1, carte_q2...) |

## Sortie
Variable `df` + fichier `df_acquisitions.pkl` contenant toutes les features nécessaires.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Repertoire contenant les CSV (relatif au notebook)
DATA_DIR = Path("data")

# ── Encodage des mois bancaires ──────────────────────────────────────────────
# Epoch : Janvier 1994 = indice 1
# Formule : indice = (annee - 1994) * 12 + mois
# Exemples : Jan 2023 = 349 | Dec 2023 = 360 | Jan 2024 = 361 | Dec 2025 = 384

EPOCH_YEAR = 1994

def date_to_idx(year, month):
    """Convertit une date (annee, mois) en indice bancaire."""
    return (year - EPOCH_YEAR) * 12 + month

def idx_to_yearmonth(idx):
    """Convertit un indice bancaire en (annee, mois)."""
    year = EPOCH_YEAR + (idx - 1) // 12
    month = (idx - 1) % 12 + 1
    return year, month

# Verifications
assert date_to_idx(2023, 1) == 349
assert date_to_idx(2025, 12) == 384
assert idx_to_yearmonth(349) == (2023, 1)
assert idx_to_yearmonth(384) == (2025, 12)
print("Encodage bancaire : OK")
print(f"  Janv. 2023 = {date_to_idx(2023,1)} | Dec. 2025 = {date_to_idx(2025,12)}")

In [ ]:
# ── Fonctions utilitaires ────────────────────────────────────────────────────

def load_csv(path):
    """Charge un CSV avec fallback d'encodage (UTF-8 -> Latin-1 -> CP1252)."""
    path = Path(path)
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc, sep=None, engine="python")
            print(f"  Charge : {path.name} ({len(df):,} lignes, encoding={enc})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossible de lire : {path}")


def detect_client_id(df):
    """Detecte la colonne identifiant client parmi les noms connus."""
    candidates = ["client_id", "clientid", "id_client", "customer_id",
                  "cust_id", "no_client", "num_client", "id"]
    for col in df.columns:
        if col.lower().strip() in candidates:
            return col
    print(f"  AVERTISSEMENT : ID client non detecte, utilisation de '{df.columns[0]}'")
    return df.columns[0]


def standardize_client_id(df, id_col):
    """Renomme, nettoie et deduplique la colonne identifiant client."""
    df = df.copy()
    df = df.rename(columns={id_col: "client_id"})
    df["client_id"] = df["client_id"].astype(str).str.strip()
    df = df[df["client_id"].notna() & (df["client_id"] != "") & (df["client_id"] != "nan")]
    df = df.drop_duplicates(subset="client_id", keep="first")
    return df


def detect_month_idx_columns(df):
    """Detecte les colonnes dont le nom est un indice de mois bancaire (entier, plage 200-500)."""
    idx_cols = []
    for col in df.columns:
        try:
            val = int(col)
            if 200 <= val <= 600:  # couvre ~1994 a ~2040
                idx_cols.append(val)
        except (ValueError, TypeError):
            pass
    return sorted(idx_cols)


print("Fonctions utilitaires : OK")

In [ ]:
# ── STOC : chargement des fichiers demographiques ────────────────────────────
print("=" * 60)
print("CHARGEMENT STOC (DEMOGRAPHICS)")
print("=" * 60)

stock_frames = []
for yy in ["23", "24", "25"]:
    path = DATA_DIR / f"STOC_{yy}.csv"
    if not path.exists():
        print(f"  AVERTISSEMENT : {path} introuvable.")
        continue
    raw = load_csv(path)
    id_col = detect_client_id(raw)
    raw = standardize_client_id(raw, id_col)
    stock_frames.append(raw)

if not stock_frames:
    raise FileNotFoundError(f"Aucun fichier STOC_YY.csv trouve dans {DATA_DIR}")

stock_df = pd.concat(stock_frames, ignore_index=True)
stock_df = stock_df.drop_duplicates(subset="client_id", keep="first")

print(f"\nColonnes STOC : {list(stock_df.columns)}")

# ── Detecter date_creation ──────────────────────────────────────────────────
date_col = None
for col in stock_df.columns:
    if any(kw in col.lower() for kw in ["date_cre", "date_ouv", "creation", "ouverture", "opened"]):
        date_col = col
        break

if date_col:
    stock_df["date_creation"] = pd.to_datetime(stock_df[date_col], errors="coerce")
    if date_col != "date_creation":
        stock_df = stock_df.drop(columns=[date_col])
    stock_df["annee_acquisition"] = stock_df["date_creation"].dt.year.astype("Int64")
    stock_df["mois_acquisition"]  = stock_df["date_creation"].dt.month.astype("Int64")
    stock_df["trimestre_acquisition"] = stock_df["date_creation"].dt.quarter.map(
        {1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"}
    )
    stock_df["acquisition_idx"] = stock_df.apply(
        lambda r: date_to_idx(int(r["annee_acquisition"]), int(r["mois_acquisition"]))
                  if pd.notna(r["annee_acquisition"]) else np.nan,
        axis=1
    )
else:
    print("  ERREUR : date_creation non trouvee. Colonnes disponibles :")
    print(f"  {list(stock_df.columns)}")

# ── Standardiser segment et profession ─────────────────────────────────────
for variants, std in [(["segment", "seg", "categorie_client", "classe"], "segment"),
                       (["profession", "csp", "categorie_socio", "job", "metier"], "profession")]:
    for v in variants:
        matches = [c for c in stock_df.columns if c.lower() == v and c != std]
        if matches:
            stock_df = stock_df.rename(columns={matches[0]: std})
            break

# ── Resume ──────────────────────────────────────────────────────────────────
print(f"\nTotal clients STOC : {len(stock_df):,}")
if "annee_acquisition" in stock_df.columns:
    print("\nClients par annee :")
    print(stock_df["annee_acquisition"].value_counts().sort_index().to_string())
if "trimestre_acquisition" in stock_df.columns:
    print("\nClients par trimestre :")
    print(stock_df.groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"].count().to_string())
if "segment" in stock_df.columns:
    print("\nSegments :")
    print(stock_df["segment"].value_counts().to_string())

In [ ]:
# ── PNB : chargement et construction de la matrice mois x client ─────────────
print("=" * 60)
print("CHARGEMENT PNB (NBI MENSUEL)")
print("=" * 60)

pnb_frames = []
all_month_idx = set()

for yy in ["23", "24", "25"]:
    path = DATA_DIR / f"PNB_{yy}.csv"
    if not path.exists():
        print(f"  AVERTISSEMENT : {path} introuvable.")
        continue
    raw = load_csv(path)
    id_col = detect_client_id(raw)
    raw = standardize_client_id(raw, id_col)

    month_cols = detect_month_idx_columns(raw)
    if not month_cols:
        print(f"  ERREUR : aucune colonne d'indice mois detectee dans PNB_{yy}")
        print(f"  Colonnes disponibles : {list(raw.columns[:20])}")
        continue

    y_min, y_max = idx_to_yearmonth(min(month_cols)), idx_to_yearmonth(max(month_cols))
    print(f"  PNB_{yy} : {len(month_cols)} mois ({y_min[0]}-{y_min[1]:02d} a {y_max[0]}-{y_max[1]:02d})")
    all_month_idx.update(month_cols)

    keep = ["client_id"] + [str(c) for c in month_cols if str(c) in raw.columns]
    pnb_frames.append(raw[keep])

if not pnb_frames:
    raise FileNotFoundError(f"Aucun fichier PNB_YY.csv trouve dans {DATA_DIR}")

# Fusionner tous les fichiers PNB : outer join pour garder tous les mois
pnb_wide = pnb_frames[0]
for frame in pnb_frames[1:]:
    pnb_wide = pd.merge(pnb_wide, frame, on="client_id", how="outer")

# Convertir les colonnes numeriques en float
month_str_cols = [str(c) for c in sorted(all_month_idx)]
for col in month_str_cols:
    if col in pnb_wide.columns:
        pnb_wide[col] = pd.to_numeric(pnb_wide[col], errors="coerce")

print(f"\nMatrice PNB : {len(pnb_wide):,} clients x {len(month_str_cols)} mois")
print(f"Plage : indice {min(all_month_idx)} ({idx_to_yearmonth(min(all_month_idx))}) "
      f"a {max(all_month_idx)} ({idx_to_yearmonth(max(all_month_idx))})")

In [ ]:
# ── Extraction pnb_m1..pnb_m12 par client selon sa date d'acquisition ────────
print("=" * 60)
print("EXTRACTION DES 12 MOIS DE PNB PAR CLIENT")
print("=" * 60)

# Fusionner stock + PNB wide
df_work = pd.merge(stock_df, pnb_wide, on="client_id", how="left")
print(f"Apres fusion STOC + PNB : {len(df_work):,} clients")

# Precalculer les colonnes PNB disponibles comme set pour O(1) lookup
available_pnb_cols = set(pnb_wide.columns) - {"client_id"}

def extract_monthly_pnb(acq_idx):
    """
    Retourne la liste des noms de colonnes pour pnb_m1..pnb_m12
    a partir de l'indice d'acquisition.
    """
    if pd.isna(acq_idx):
        return [None] * 12
    return [str(int(acq_idx) + m - 1) for m in range(1, 13)]

# Construire les colonnes pnb_m1..pnb_m12
print("Construction des colonnes pnb_m1..pnb_m12 ...")
monthly_data = {}
for m in range(1, 13):
    col_name = f"pnb_m{m}"
    # Pour chaque client, recuperer le mois acquisition_idx + (m-1)
    monthly_data[col_name] = df_work.apply(
        lambda r: r.get(str(int(r["acquisition_idx"]) + m - 1), np.nan)
                  if pd.notna(r.get("acquisition_idx")) else np.nan,
        axis=1
    )

df_work = pd.concat([df_work, pd.DataFrame(monthly_data, index=df_work.index)], axis=1)

# Supprimer les colonnes indices brutes
df_work = df_work.drop(columns=[c for c in month_str_cols if c in df_work.columns])

# Calcul des cumuls
df_work["pnb_cumul_3m"]  = df_work[[f"pnb_m{i}" for i in range(1, 4)]].sum(axis=1, min_count=1)
df_work["pnb_cumul_6m"]  = df_work[[f"pnb_m{i}" for i in range(1, 7)]].sum(axis=1, min_count=1)
df_work["pnb_12m_total"] = df_work[[f"pnb_m{i}" for i in range(1, 13)]].sum(axis=1, min_count=1)

# Couverture
has_12m = df_work[[f"pnb_m{i}" for i in range(1, 13)]].notna().all(axis=1)
has_6m  = df_work[[f"pnb_m{i}" for i in range(1, 7)]].notna().all(axis=1)
has_3m  = df_work[[f"pnb_m{i}" for i in range(1, 4)]].notna().all(axis=1)
print(f"Clients avec 12 mois complets : {has_12m.sum():,} ({has_12m.mean()*100:.1f}%)")
print(f"Clients avec  6 mois complets : {has_6m.sum():,} ({has_6m.mean()*100:.1f}%)")
print(f"Clients avec  3 mois complets : {has_3m.sum():,} ({has_3m.mean()*100:.1f}%)")
print(f"\nPNB 12m — moy: {df_work['pnb_12m_total'].mean():,.0f} EUR | "
      f"mediane: {df_work['pnb_12m_total'].median():,.0f} EUR")

In [ ]:
# ── EQUIPEMENTS : produits et flux trimestriels ───────────────────────────────
print("=" * 60)
print("CHARGEMENT EQUIPEMENTS")
print("=" * 60)

equip_path = DATA_DIR / "clients_equipement_full.csv"

if equip_path.exists():
    equip_raw = load_csv(equip_path)
    id_col = detect_client_id(equip_raw)
    equip_raw = standardize_client_id(equip_raw, id_col)
    print(f"Colonnes equipement : {list(equip_raw.columns)}")

    # ── Detecter les produits (colonnes avec suffixe _q1/_q2/_q3/_q4) ─────────
    PRODUCT_KW = ["compte", "carte", "epargne", "credit", "assurance",
                  "livret", "prevoyance", "pret", "hypothe", "bourse",
                  "vie", "deces", "invalidite", "retraite"]
    QUARTERS = ["_q1", "_q2", "_q3", "_q4"]

    # Construire le mapping base_product -> {q1: col, q2: col, q3: col, q4: col}
    product_map = {}
    for col in equip_raw.columns:
        col_lower = col.lower()
        for q in QUARTERS:
            if col_lower.endswith(q):
                base = col_lower[:-len(q)].rstrip("_")
                if any(kw in base for kw in PRODUCT_KW):
                    if base not in product_map:
                        product_map[base] = {}
                    product_map[base][q.lstrip("_")] = col
                break

    print(f"\nProduits detectes ({len(product_map)}) : {sorted(product_map.keys())}")

    equip_feat = equip_raw[["client_id"]].copy()

    for base, qcols in sorted(product_map.items()):
        # has_X : 1 si quantite > 0 dans Q1 ou Q2 (6 premiers mois)
        early_cols = [qcols[q] for q in ["q1", "q2"] if q in qcols]
        if early_cols:
            equip_feat[f"has_{base}"] = (
                equip_raw[early_cols].fillna(0).gt(0).any(axis=1).astype(int)
            )
        # Evolution Q1 -> Q4 (signal de cross-sell)
        if "q1" in qcols and "q4" in qcols:
            q1_qty = pd.to_numeric(equip_raw[qcols["q1"]], errors="coerce").fillna(0)
            q4_qty = pd.to_numeric(equip_raw[qcols["q4"]], errors="coerce").fillna(0)
            equip_feat[f"{base}_evolution"] = q4_qty - q1_qty

    # n_produits_total
    has_cols = [c for c in equip_feat.columns if c.startswith("has_")]
    equip_feat["n_produits_total"] = equip_feat[has_cols].sum(axis=1) if has_cols else 0

    # 4 familles de produits
    FAMILIES = {
        "fam_banque_quotidienne":      ["compte", "carte"],
        "fam_epargne_investissement":  ["epargne", "livret", "vie", "bourse", "retraite"],
        "fam_credit":                  ["credit", "pret", "hypothe"],
        "fam_assurance_protection":    ["assurance", "prevoyance", "deces", "invalidite"],
    }
    for fam_col, keywords in FAMILIES.items():
        fam_has = [c for c in has_cols if any(kw in c for kw in keywords)]
        equip_feat[fam_col] = equip_feat[fam_has].max(axis=1) if fam_has else 0
    equip_feat["n_familles_produits"] = equip_feat[list(FAMILIES.keys())].sum(axis=1)

    # Colonnes de flux (inflow / outflow / solde) si disponibles
    FLOW_KW = ["entree", "sortie", "inflow", "outflow", "solde", "encours", "balance", "flux"]
    for col in equip_raw.columns:
        if any(kw in col.lower() for kw in FLOW_KW) and col != "client_id":
            equip_feat[col] = pd.to_numeric(equip_raw[col], errors="coerce")

    print(f"\nFeatures equipement creees ({len(equip_feat.columns)-1}) :")
    print([c for c in equip_feat.columns if c != "client_id"])
    print(f"\nDistribution n_produits_total :")
    print(equip_feat["n_produits_total"].value_counts().sort_index().to_string())

else:
    print(f"AVERTISSEMENT : {equip_path} introuvable.")
    print("Les features has_* seront absentes — les analyses HA5 seront limitees.")
    equip_feat = None

In [ ]:
# ── Fusion des 3 sources ─────────────────────────────────────────────────────
print("=" * 60)
print("FUSION DES SOURCES")
print("=" * 60)

df = df_work.copy()
print(f"Apres STOC + PNB : {len(df):,} clients")

if equip_feat is not None:
    df = pd.merge(df, equip_feat, on="client_id", how="left")
    n_equip = df["n_produits_total"].notna().sum()
    print(f"Apres fusion equipement : {len(df):,} clients | "
          f"avec equipement : {n_equip:,} ({n_equip/len(df)*100:.1f}%)")

# Verifier doublons
n_dup = df.duplicated(subset="client_id").sum()
if n_dup > 0:
    print(f"  AVERTISSEMENT : {n_dup} doublons detectes -> deduplication")
    df = df.drop_duplicates(subset="client_id", keep="first")

print(f"\nDataFrame fusionne : {df.shape[0]:,} clients x {df.shape[1]} colonnes")

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# ── produits_ajoutes : cross-sell dans les 12 premiers mois ─────────────────
evo_cols = [c for c in df.columns if c.endswith("_evolution")]
if evo_cols:
    df["produits_ajoutes"] = df[evo_cols].clip(lower=0).sum(axis=1)
    print(f"produits_ajoutes — moy: {df['produits_ajoutes'].mean():.2f}")

# ── Trajectoire PNB sur 6 mois (pour HA6) ────────────────────────────────────
pnb_6m_cols = [f"pnb_m{i}" for i in range(1, 7)]
available_6m = [c for c in pnb_6m_cols if c in df.columns]

if len(available_6m) >= 3:
    from sklearn.linear_model import LinearRegression

    def compute_pnb_slope(row):
        vals = pd.array([row[c] for c in available_6m], dtype="float64")
        ok = ~np.isnan(vals)
        if ok.sum() < 2:
            return 0.0
        x = np.where(ok)[0].reshape(-1, 1)
        y = vals[ok]
        return LinearRegression().fit(x, y).coef_[0]

    df["pnb_pente"]               = df.apply(compute_pnb_slope, axis=1)
    df["pnb_volatilite_mensuelle"] = df[available_6m].std(axis=1)
    df["mois_pnb_positif"]         = (df[available_6m] > 0).sum(axis=1)
    print(f"pnb_pente — moy: {df['pnb_pente'].mean():.2f}")
    print(f"mois_pnb_positif — moy: {df['mois_pnb_positif'].mean():.1f}")

    # Slow starters et early peaks
    if available_6m[0] in df.columns and available_6m[2] in df.columns:
        m1, m3 = available_6m[0], available_6m[2]
        df["is_slow_starter"] = ((df[m1] < df[m1].median()) &
                                  (df[m3] > df[m3].median())).astype(int)
        df["is_early_peak"]   = ((df[m1] > df[m1].quantile(0.75)) &
                                  (df["pnb_pente"] < -20)).astype(int)
        print(f"is_slow_starter : {df['is_slow_starter'].sum():,} ({df['is_slow_starter'].mean()*100:.1f}%)")

# ── Anciennete en mois a la date de reference ─────────────────────────────────
REFERENCE_DATE = pd.Timestamp("2026-03-31")
if "date_creation" in df.columns:
    df["mois_anciennete"] = ((REFERENCE_DATE - df["date_creation"]).dt.days / 30.44).round(0)

# ── Nettoyage colonnes intermediaires ────────────────────────────────────────
df = df.drop(columns=[c for c in ["acquisition_idx", "mois_acquisition", "_source_stoc"]
                       if c in df.columns])
df = df.drop(columns=[c for c in df.columns if c.endswith("_evolution")])

print(f"\nShape apres feature engineering : {df.shape}")

In [ ]:
# ── Filtrage et classification des clients ────────────────────────────────────
print("=" * 60)
print("CLASSIFICATION : ENTRAINEMENT vs SCORING")
print("=" * 60)

# Eligible a l'entrainement : acquisition <= mars 2025
# (12 mois de PNB observables jusqu'a mars 2026)
CUTOFF = pd.Timestamp("2025-03-31")

if "date_creation" in df.columns:
    df["est_eligible_training"] = df["date_creation"] <= CUTOFF
    n_train = df["est_eligible_training"].sum()
    n_score = (~df["est_eligible_training"]).sum()
    print(f"Eligible entrainement (acquisition <= mars 2025) : {n_train:,}")
    print(f"Fenetre de scoring (acquisition > mars 2025)    : {n_score:,}")

    if "annee_acquisition" in df.columns and "trimestre_acquisition" in df.columns:
        print("\nCohorts eligibles :")
        cohorts = (
            df[df["est_eligible_training"]]
            .groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"]
            .count()
            .reset_index()
        )
        cohorts.columns = ["annee", "trimestre", "n_clients"]
        print(cohorts.to_string(index=False))

In [ ]:
# ── Resume final et export ────────────────────────────────────────────────────
print("=" * 60)
print("RESUME FINAL")
print("=" * 60)

print(f"Shape : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

print("\nColonnes et taux de completude :")
for col in df.columns:
    n_null = df[col].isna().sum()
    pct = n_null / len(df) * 100
    flag = "  <<< ATTENTION" if pct > 30 else ""
    print(f"  {col:<42} {str(df[col].dtype):<10} null={pct:5.1f}%{flag}")

if "pnb_12m_total" in df.columns:
    print("\nPNB 12 mois (clients eligibles entrainement) :")
    train_pnb = df.loc[df["est_eligible_training"], "pnb_12m_total"] if "est_eligible_training" in df.columns else df["pnb_12m_total"]
    print(train_pnb.describe().round(0).to_string())

# Export pickle
out_path = Path("df_acquisitions.pkl")
with open(out_path, "wb") as f:
    pickle.dump(df, f)
print(f"\nExporte : {out_path.resolve()}")
print("Pour charger dans les notebooks HA1-HA10 :")
print("  import pickle")
print("  df = pickle.load(open('df_acquisitions.pkl', 'rb'))")

In [ ]:
# ── Apercu graphique ──────────────────────────────────────────────────────────
print("=" * 60)
print("APERCU GRAPHIQUE")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Volume d'acquisition par trimestre
if "annee_acquisition" in df.columns and "trimestre_acquisition" in df.columns:
    vol = (
        df.groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"]
        .count()
        .reset_index()
    )
    vol["label"] = vol["annee_acquisition"].astype(str) + "-" + vol["trimestre_acquisition"]
    axes[0, 0].bar(vol["label"], vol["client_id"], color="steelblue", edgecolor="white")
    axes[0, 0].set_title("Nouveaux clients par trimestre d'acquisition")
    axes[0, 0].set_ylabel("Nombre de clients")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Distribution PNB 12 mois (clients eligibles)
if "pnb_12m_total" in df.columns:
    mask = df["est_eligible_training"] if "est_eligible_training" in df.columns else pd.Series([True]*len(df))
    pnb_vals = df.loc[mask, "pnb_12m_total"].dropna()
    p99 = pnb_vals.quantile(0.99)
    axes[0, 1].hist(pnb_vals.clip(upper=p99), bins=60, color="teal", edgecolor="white", alpha=0.8)
    axes[0, 1].set_title("Distribution PNB 12 mois (clients eligibles)")
    axes[0, 1].set_xlabel("PNB 12 mois (EUR)")
    axes[0, 1].set_ylabel("Frequence")
    axes[0, 1].grid(alpha=0.3)

# 3. PNB moyen par segment
if "segment" in df.columns and "pnb_12m_total" in df.columns:
    seg_means = df.groupby("segment")["pnb_12m_total"].mean().sort_values(ascending=True)
    seg_means.plot(kind="barh", ax=axes[1, 0], color="coral", edgecolor="white")
    axes[1, 0].set_title("PNB moyen 12 mois par segment")
    axes[1, 0].set_xlabel("PNB moyen (EUR)")
    axes[1, 0].grid(axis="x", alpha=0.3)

# 4. Distribution nombre de produits
if "n_produits_total" in df.columns:
    prod_dist = df["n_produits_total"].value_counts().sort_index()
    axes[1, 1].bar(prod_dist.index.astype(str), prod_dist.values, color="mediumorchid", edgecolor="white")
    axes[1, 1].set_title("Distribution du nombre de produits")
    axes[1, 1].set_xlabel("Nombre de produits")
    axes[1, 1].set_ylabel("Nombre de clients")
    axes[1, 1].grid(axis="y", alpha=0.3)

plt.suptitle("Preparation des donnees — Modele CLV Nouvelles Acquisitions", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDonnees pretes. Lancer les notebooks HA1 a HA10 pour les analyses d'hypotheses.")